# Zero-Shot vs Few-Shot Prompting

## Objective

In this practice, we will compare zero-shot and few-shot prompting
for the same classification task.

We will evaluate both approaches based on:

- Output quality
- Instruction following
- Consistency
- Accuracy

The goal is to understand when examples can improve an LLM's
performance without changing the model's parameters.

## Step 2  Model/API Setup

For a practical prompt-engineering experiment, we need an LLM that we can send prompts to from Colab.

Since the objective is specifically zero-shot vs few-shot prompting, we don't need to train or fine-tune anything.

We'll use the Google Gemini API from Colab.

### 2.1 Install the SDK

In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.0 which is incompatible.


### 2.2 Add your API key securely

In [8]:
from google import genai

api_key = "AQ.Ab8RN6IPKUfIdoAcrVc4sonRgOxKflzlZFxW8rxC2eaIhMXTdQ"

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


### 2.3 Test the LLM

In [10]:
response = client.interactions.create(
    model="gemini-3.8-flash",
    input="Explain what an LLM is in one simple sentence."
)

print(response.output_text)

A Large Language Model (LLM) is an AI program trained on vast amounts of text to understand, predict, and generate human-like language.




We successfully connected Google Gemini to the Colab notebook and
sent our first prompt to the language model.

The **input** parameter contains the prompt given to the model.
The model processes the instruction and returns generated text.

For this experiment, we will use the same model for both zero-shot
and few-shot prompting so that the main variable is the prompting
strategy rather than the model.

We are using the Gemini Interactions API with the Gemini 3.8 Flash
model.

## Step 3  Zero-Shot Prompting

### 3.1 Zero-Shot Prompting

Zero-shot prompting means asking an LLM to perform a task without
providing examples of the expected input and output.

The model receives only the instructions and task requirements.

In this experiment, we will use zero-shot prompting for a
customer-support message classification task.

The model will classify each message into one of three categories:

- Billing
- Technical
- Account

No examples will be provided in the zero-shot prompt.

### Step 3.2  Create the Zero-Shot Prompt

Now we create the actual prompt. The important point is that we will not include any examples.

In [11]:
zero_shot_prompt = """
You are a customer-support message classifier.

Classify the customer message into exactly one of these categories:

- Billing: payment, charges, refunds, invoices, or subscription costs.
- Technical: software errors, bugs, crashes, or features not working.
- Account: login, password, profile, account access, or account settings.

Return only the category name. Do not provide an explanation.

Customer message:
I was charged twice for the same subscription.
"""

### Step 3.3  Send the Prompt to Gemini

In [12]:
response = client.interactions.create(
    model="gemini-3.8-flash",
    input=zero_shot_prompt
)

zero_shot_output = response.output_text

print(zero_shot_output)

Billing


### Observation

The zero-shot prompt successfully classified the customer message
as **Billing**.

The model was able to perform the task using only the instructions
and category definitions without seeing any examples.

This result provides our baseline for comparison with few-shot
prompting.

## Step 4  Few-Shot Prompting


### 4.1 Few-Shot Prompting

### Concept

Few-shot prompting means providing an LLM with a small number of
examples that demonstrate how a task should be performed.

The examples contain an input and its expected output.

In this experiment, we will use the same customer-support
classification task used in the zero-shot experiment.

The only major difference is that the few-shot prompt includes
examples for Billing, Technical, and Account.

This allows us to compare the effect of examples on model output.

### 4.2 Create the Few-Shot Prompt

In [13]:
few_shot_prompt = """
You are a customer-support message classifier.

Classify the customer message into exactly one of these categories:

- Billing: payment, charges, refunds, invoices, or subscription costs.
- Technical: software errors, bugs, crashes, or features not working.
- Account: login, password, profile, account access, or account settings.

Return only the category name. Do not provide an explanation.

Examples:

Customer message:
I do not recognize the payment on my credit card.
Category:
Billing

Customer message:
The application crashes whenever I upload a file.
Category:
Technical

Customer message:
I forgot my password and cannot log in.
Category:
Account

Now classify this customer message:

Customer message:
I was charged twice for the same subscription.
"""

### 4.3 Send the Few-Shot Prompt to Gemini

In [14]:
response = client.interactions.create(
    model="gemini-3.8-flash",
    input=few_shot_prompt
)

few_shot_output = response.output_text

print(few_shot_output)

Billing


### Observation

The few-shot prompt provided examples before asking the model to
classify the new customer message.

The examples demonstrated the relationship between customer
messages and their expected categories.

The model classified the test message as **Billing**.

Compared with zero-shot prompting, few-shot prompting gives the
model additional task-specific context through examples.

## Step 5 Evaluation Dataset



### 5.1 Test Dataset

To compare zero-shot and few-shot prompting fairly, both approaches
will be evaluated on the same set of customer-support messages.

Each message has a predefined expected category.

The three categories are:

- Billing
- Technical
- Account

Using the same test dataset allows us to compare both prompting
methods under the same conditions.

In [23]:
test_data = [
    {
        "message": "I was charged twice for my monthly plan.",
        "expected": "Billing"
    },
    {
        "message": "The application freezes whenever I open the dashboard.",
        "expected": "Technical"
    },
    {
        "message": "I cannot remember my password and need to sign in.",
        "expected": "Account"
    },
    {
        "message": "How can I get a refund for my subscription?",
        "expected": "Billing"
    },
    {
        "message": "The upload feature gives me an error every time.",
        "expected": "Technical"
    },
    {
        "message": "I want to change the email address linked to my account.",
        "expected": "Account"
    },
    {
        "message": "Why was my subscription payment processed twice?",
        "expected": "Billing"
    },
    {
        "message": "The website stops responding when I click the login button.",
        "expected": "Technical"
    },
    {
        "message": "My account is locked and I cannot access it.",
        "expected": "Account"
    }
]

print(f"Number of test cases: {len(test_data)}")

Number of test cases: 9


### 5.2 Prepare Expected Labels

In [24]:
expected_labels = [item["expected"] for item in test_data]

print("Expected labels:")
print(expected_labels)

Expected labels:
['Billing', 'Technical', 'Account', 'Billing', 'Technical', 'Account', 'Billing', 'Technical', 'Account']


### 5.3 Evaluation Functions

Now we create reusable evaluation functions.

Concept

We need to measure more than just accuracy.

We will check:

1. **Accuracy:** How many predictions are correct?
2. **Format compliance:**  Did the model return a valid category?
3. **Consistency:**  Did the model produce stable outputs?

In [25]:
VALID_LABELS = {"billing", "technical", "account"}


def normalize_label(label):
    """Normalize model output for comparison."""
    return label.strip().lower()


def calculate_accuracy(expected, predicted):
    """Calculate classification accuracy."""
    if len(expected) != len(predicted):
        raise ValueError("Expected and predicted lists must have the same length.")

    correct = sum(
        normalize_label(exp) == normalize_label(pred)
        for exp, pred in zip(expected, predicted)
    )

    return correct / len(expected)


def check_format_compliance(predicted):
    """Check whether every prediction is a valid category."""
    valid = 0

    for label in predicted:
        if normalize_label(label) in VALID_LABELS:
            valid += 1

    return valid / len(predicted)


def show_results(method, expected, predicted):
    """Display evaluation results."""
    accuracy = calculate_accuracy(expected, predicted)
    format_score = check_format_compliance(predicted)

    print(f"Method: {method}")
    print(f"Accuracy: {accuracy:.2%}")
    print(f"Format Compliance: {format_score:.2%}")

### 5.4 Prepare Result Storage

Because the API quota is currently exhausted, we will create the result structure now and fill it with the real model outputs later

In [26]:
zero_shot_results = []
few_shot_results = []

print("Result containers created.")

Result containers created.


### 5.5 Example Evaluation Structure

For now, we can verify that our evaluation system works using a temporary test prediction list.

In [27]:
demo_predictions = [
    "Billing",
    "Technical",
    "Account",
    "Billing",
    "Technical",
    "Account",
    "Billing",
    "Technical",
    "Account"
]

show_results(
    "Demo Evaluation",
    expected_labels,
    demo_predictions
)

Method: Demo Evaluation
Accuracy: 100.00%
Format Compliance: 100.00%


### 5.6 Compare Zero-Shot and Few-Shot

In [28]:
comparison = {
    "Zero-Shot": {
        "accuracy": None,
        "format_compliance": None
    },
    "Few-Shot": {
        "accuracy": None,
        "format_compliance": None
    }
}

comparison

{'Zero-Shot': {'accuracy': None, 'format_compliance': None},
 'Few-Shot': {'accuracy': None, 'format_compliance': None}}

### 5.7 Consistency Evaluation

For consistency, the same prompt should ideally produce the same category when run multiple times.

However, do not run this now, because every API call consumes quota.

In [29]:
zero_shot_repeated = [
    "Billing",
    "Billing",
    "Billing"
]

In [30]:
def calculate_consistency(predictions):
    """Calculate the proportion of the most common prediction."""
    if not predictions:
        return 0

    normalized = [normalize_label(p) for p in predictions]

    most_common_count = max(
        normalized.count(label)
        for label in set(normalized)
    )

    return most_common_count / len(normalized)

In [31]:
predictions = ["Billing", "Billing", "Billing"]

consistency = calculate_consistency(predictions)

print(f"Consistency: {consistency:.2%}")

Consistency: 100.00%


## 6. Final Comparison

The zero-shot and few-shot approaches are compared using the same
test dataset and the same classification task.

The main comparison criteria are:

- Accuracy
- Format compliance
- Consistency
- Amount of task-specific guidance

In [32]:
comparison_data = {
    "Criteria": [
        "Instructions",
        "Examples",
        "Accuracy",
        "Format Compliance",
        "Consistency"
    ],
    "Zero-Shot": [
        "Instructions only",
        "No examples",
        "Pending",
        "Pending",
        "Pending"
    ],
    "Few-Shot": [
        "Instructions + examples",
        "3 examples",
        "Pending",
        "Pending",
        "Pending"
    ]
}

import pandas as pd

comparison_df = pd.DataFrame(comparison_data)

comparison_df

,Criteria,Zero-Shot,Few-Shot
0,Instructions,Instructions only,Instructions + examples
1,Examples,No examples,3 examples
2,Accuracy,Pending,Pending
3,Format Compliance,Pending,Pending
4,Consistency,Pending,Pending


## 7. Final Conclusion

Zero-shot prompting performs a task using instructions without
providing examples. It is simple and requires less prompt content.

Few-shot prompting adds a small number of examples to demonstrate
the expected task behavior. These examples can provide additional
context and help the model understand the required output pattern.

In this experiment, both approaches use the same model, the same
classification categories, and the same test dataset.

The final accuracy and consistency results will be added after the
remaining API evaluations are completed.

The main practical lesson is that prompting strategy can influence
LLM behavior without changing the model itself. The choice between
zero-shot and few-shot prompting depends on the complexity of the
task and how much guidance the model needs.